# Bronze Layer — Metadata Ingestion
**Layer:** Bronze (raw ingestion — no transformations applied)
**Source:** AWS S3 — `s3://test-capstone-project-metadata/metadata_realistic_10k.csv`

In [ ]:
# Catalog widget — injected by Job base_parameters or set manually
dbutils.widgets.text("catalog", "metadata_governance", "Catalog")
catalog = dbutils.widgets.get("catalog")
print(f"Using catalog: {catalog}")

## Step 1 — Read Raw CSV from S3

In [ ]:
from pyspark.sql.functions import current_timestamp, current_date, lit

df_bronze = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("s3://test-capstone-project-metadata/metadata_realistic_10k.csv")

## Step 2 — Add Audit Columns

In [ ]:
df_bronze = df_bronze \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("load_date", current_date()) \
    .withColumn("record_source", lit("AWS_S3"))

## Step 3 — Save to Bronze Delta Table

In [ ]:
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.bronze.bronze_metadata")

## Step 4 — Validate Row Count

In [ ]:
bronze_df = spark.table(f"{catalog}.bronze.bronze_metadata")
row_count = bronze_df.count()
print(f"Bronze Row Count: {row_count}")
assert row_count > 0, "Bronze table is empty — ingestion failed"

## Step 5 — Validate Schema

In [ ]:
bronze_df.printSchema()

## Step 6 — Validate Audit Columns

In [ ]:
display(
    bronze_df.select("ingestion_timestamp", "load_date", "record_source").limit(10)
)

## Step 7 — Validate Data Completeness

In [ ]:
from pyspark.sql.functions import col, count, when, isnull

null_counts = bronze_df.select(
    [count(when(isnull(c) | (col(c).cast("string") == ""), c)).alias(c)
     for c in bronze_df.columns]
)
display(null_counts)